In [4]:
import pandas as pd

In [5]:
%pip install -q datasets evaluate rouge_score bert_score sacrebleu transformers accelerate

In [6]:
from datasets import load_dataset

dataset = load_dataset("Nan-Do/instructional_code-search-net-python", split="train")
df = dataset.to_pandas()

In [7]:
df = df.dropna(subset=["INSTRUCTION", "RESPONSE"])
df = df[(df["INSTRUCTION"].str.strip() != "") & (df["RESPONSE"].str.strip() != "")]
df = df[df["INSTRUCTION"].str.contains("explain|explanation|what .* doing|what does .* do", case=False, na=False)]

In [8]:
from sklearn.model_selection import train_test_split

train_df, eval_df = train_test_split(df, test_size=0.1, random_state=42)
test_data = eval_df.sample(n=150, random_state=42)

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("lalit63/gemma-3-1b-python-explainer")
model = AutoModelForCausalLM.from_pretrained("lalit63/gemma-3-1b-python-explainer", device_map="auto")

config.json:   0%|          | 0.00/1.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/705 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

In [10]:
model_answers = []

for question in test_data["INSTRUCTION"]:
    prompt = "### Instruction:\n" + question + "\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to("cuda")
    output = model.generate(**inputs, max_new_tokens=60)
    answer = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    model_answers.append(answer)

In [11]:
import evaluate

real_answers = test_data["RESPONSE"].tolist()

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")
bertscore = evaluate.load("bertscore")

print("ROUGE-L:", rouge.compute(predictions=model_answers, references=real_answers)["rougeL"])
print("BLEU:", bleu.compute(predictions=model_answers, references=[[r] for r in real_answers])["score"])
print("BERTScore:", sum(bertscore.compute(predictions=model_answers, references=real_answers, lang="en")["f1"]) / 150)

ROUGE-L: 0.30423486436157676
BLEU: 7.810368755254251


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore: 0.8725405605634053


In [12]:
for i in range(5):
    print("REAL:", real_answers[i])
    print("MODEL:", model_answers[i])
    print("-" * 40)

REAL: Adjusts the dates of the first and last time series to fix resampling errors.
MODEL:             fresult = first - offset.as_timedelta()
        else:
            fresult = first - offset.as_timedelta()

        if not fresult.total_seconds():
            # FIXME add a way to determine the difference between the offset
            # and the time
----------------------------------------
REAL: Return the maximum of self. See Also
        numpy. amax
        min
       
MODEL: Return the maximum absolute value of the elements in the sequence.
----------------------------------------
REAL: Get the revision of a changset.
MODEL:  is None:
                raise UnknownReferenceError("There is an unknown revision %s"
                    " in the repository" % (revision))

            return _ref_revision

        elif is_bstr and revision is None or revision == "":
            raise InvalidReferenceError("The revision
----------------------------------------
REAL: Return all events that